# Deal Review — Version Jev

Run from this folder after configuring `.env` and starting Ollama.
Every deal passes through Jev: `HUMAN_REVIEW` or `GENERATE_REPORT`.
The panel supplies the loan decision; a human can override it.


## Pipeline

```text
PDF → Extractor → Compliance / Financial / Risk (parallel) → Synthesizer → Jev
                                                                           ├─ GENERATE_REPORT → Report
                                                                           └─ HUMAN_REVIEW → Pause → Human decision → Report
```

Jev routing is defined in `jev_review.py`. Low confidence, unresolved panel
escalation, missing keys, and API errors also require human review.


## Step 1 — Install

In [ ]:
%pip install -q -r requirements.txt


## Step 2 — Imports

In [ ]:
import operator
import json
import uuid
import datetime
from jev_review import node_jev_review, route_post_jev

from typing import Annotated, TypedDict, Literal

from langchain_ollama import ChatOllama

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

from langgraph.graph import StateGraph, START, END

from langgraph.types import Send, interrupt, Command

from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

from pydantic import BaseModel, Field

from pypdf import PdfReader


## Step 3 — Pydantic Schemas

All data flowing through the pipeline is **typed and validated**. No more regex JSON parsing.

In [ ]:
class DealTerms(BaseModel):
    borrower:      str   = Field(description="Borrower / obligor name")
    amount_usd:    float = Field(description="Total loan amount in USD")
    ltv:           float = Field(description="Loan-to-value ratio as decimal (e.g. 0.75)")
    dti:           float = Field(description="Debt-to-income ratio as decimal")
    credit_score:  int   = Field(description="FICO / credit score 300-850")
    rate_pct:      float = Field(description="Annual interest rate percentage")
    term_months:   int   = Field(description="Loan term in months")
    jurisdiction:  str   = Field(description="Borrower jurisdiction or state")
    property_type: str   = Field(description="Asset or property type")
    purpose:       str   = Field(description="Stated loan purpose")

class ComplianceFindings(BaseModel):
    status:           Literal["PASS", "FAIL", "CONDITIONAL"]
    failed_rules:     list[str] = Field(default_factory=list)
    regulatory_notes: list[str] = Field(default_factory=list)
    confidence:       float     = Field(ge=0.0, le=1.0)
    reasoning:        str

class FinancialFindings(BaseModel):
    ltv_assessment:  str
    dti_assessment:  str
    rate_assessment: str
    financial_health: Literal["STRONG", "ADEQUATE", "WEAK", "CRITICAL"]
    strengths:       list[str] = Field(default_factory=list)
    concerns:        list[str] = Field(default_factory=list)
    confidence:      float     = Field(ge=0.0, le=1.0)
    reasoning:       str

class RiskFindings(BaseModel):
    credit_risk_score:        float = Field(ge=0, le=100)
    concentration_risk_score: float = Field(ge=0, le=100)
    composite_risk_score:     float = Field(ge=0, le=100)
    risk_level:               Literal["LOW", "MEDIUM", "HIGH", "CRITICAL"]
    mitigants:                list[str] = Field(default_factory=list)
    confidence:               float     = Field(ge=0.0, le=1.0)
    reasoning:                str

class PanelDecision(BaseModel):
    decision:            Literal["APPROVE", "CONDITIONAL_APPROVE", "REJECT", "ESCALATE"]
    overall_risk_score:  float     = Field(ge=0, le=100)
    requires_escalation: bool
    conditions:          list[str] = Field(default_factory=list)
    key_risks:           list[str] = Field(default_factory=list)
    executive_summary:   str
    panel_confidence:    float     = Field(ge=0.0, le=1.0)


## Step 4 — Configuration + LLM + Rules

In [ ]:
MODEL = "gemma4:e4b"
llm   = ChatOllama(model=MODEL, temperature=0, num_ctx=8192)

RULES = {
    "max_loan_usd":          50_000_000,
    "max_ltv":               0.80,
    "min_credit_score":      620,
    "max_dti":               0.43,
    "max_rate_pct":          18.0,
    "escalation_usd":        10_000_000,
    "blocked_jurisdictions": ["OFAC", "FATF-blacklist"],
}

print(f"LLM  : {MODEL}")
print(f"Rules: {len(RULES)} thresholds loaded")


## Step 5 — Tools with Pydantic Input Schemas

Risk tools declare an `args_schema` — the LLM sees the field descriptions and validates before calling.
Compliance needs no tools — it runs pure Python if/else against the RULES dict.

In [ ]:
class CreditRiskInput(BaseModel):
    credit_score: int   = Field(description="FICO credit score 300-850")
    ltv:          float = Field(description="Loan-to-value ratio 0-1")
    dti:          float = Field(description="Debt-to-income ratio 0-1")
    rate_pct:     float = Field(description="Annual interest rate percentage")

@tool(args_schema=CreditRiskInput)
def calc_credit_risk(credit_score: int, ltv: float, dti: float, rate_pct: float) -> str:
    """Calculate composite credit risk score 0–100 (higher = riskier). Use with borrower metrics."""
    score_pen = max(0.0, (750 - credit_score) / 130) * 40
    ltv_pen   = max(0.0, (ltv   - 0.60) / 0.20)      * 25
    dti_pen   = max(0.0, (dti   - 0.35) / 0.08)      * 20
    rate_pen  = max(0.0, (rate_pct - 6.0) / 12.0)    * 15
    score     = round(min(100.0, score_pen + ltv_pen + dti_pen + rate_pen), 1)
    return json.dumps({"credit_risk_score": score,
                       "breakdown": {"credit": round(score_pen,1), "ltv": round(ltv_pen,1),
                                     "dti": round(dti_pen,1), "rate": round(rate_pen,1)}})


class ConcentrationRiskInput(BaseModel):
    amount_usd:          float = Field(description="Loan amount in USD")
    portfolio_limit_usd: float = Field(default=500_000_000, description="Portfolio limit in USD")

@tool(args_schema=ConcentrationRiskInput)
def calc_concentration_risk(amount_usd: float, portfolio_limit_usd: float = 500_000_000) -> str:
    """Calculate concentration risk as percentage of total portfolio."""
    pct   = (amount_usd / portfolio_limit_usd) * 100
    score = round(min(100.0, pct * 5), 1)
    tier  = "HIGH" if score > 60 else "MEDIUM" if score > 30 else "LOW"
    return json.dumps({"concentration_pct": round(pct, 2),
                       "concentration_risk_score": score, "tier": tier})


RISK_TOOLS = [calc_credit_risk, calc_concentration_risk]


## Step 6 — State

`specialist_results` uses **`operator.add`** as a reducer — each parallel agent appends its result, and they're merged automatically when the fan-in fires.

In [ ]:
class SpecialistResult(TypedDict):
    specialist: str
    findings:   dict
    confidence: float

class DealState(TypedDict):
    deal_id:   str
    deal_text: str

    terms: DealTerms | None

    specialist_results: Annotated[list[SpecialistResult], operator.add]

    panel_decision:      PanelDecision | None
    requires_escalation: bool

    human_decision: str | None
    human_notes:    str | None

    jev_review: dict | None

    final_report: dict | None


## Step 7 — Nodes

### 7a — Extractor (structured output)

In [ ]:
def node_extract(state: DealState) -> dict:
    print("\n📄 [Extractor] Parsing deal document...")

    structured = llm.with_structured_output(DealTerms)

    terms = structured.invoke([
        SystemMessage(content=(
            "You are a financial document analyst. Extract all loan terms precisely. "
            "Ratios must be decimals (0.75 not 75%). If a field is missing, make a "
            "reasonable inference from context."
        )),
        HumanMessage(content=f"Extract terms from:\n\n{state['deal_text'][:6000]}"),
    ])

    print(f"   ✓ Borrower : {terms.borrower}")
    print(f"   ✓ Amount   : ${terms.amount_usd:,.0f}")
    print(f"   ✓ LTV      : {terms.ltv:.1%}  DTI: {terms.dti:.1%}  Credit: {terms.credit_score}")

    return {
        "terms": terms,
        "specialist_results": [],
        "requires_escalation": False,
        "human_decision": None,
        "human_notes": None,
        "jev_review": None,
        "panel_decision": None,
        "final_report": None,
    }


### 7b — Parallel Specialist Nodes

In [ ]:
def node_compliance_specialist(state: DealState) -> dict:
    print("\n⚖️  [Compliance] Checking rules...")
    terms = state["terms"]

    failed_rules     = []
    regulatory_notes = []

    if terms.ltv > RULES["max_ltv"]:
        failed_rules.append(f"LTV {terms.ltv:.1%} exceeds max {RULES['max_ltv']:.0%}")

    if terms.dti > RULES["max_dti"]:
        failed_rules.append(f"DTI {terms.dti:.1%} exceeds max {RULES['max_dti']:.0%}")

    if terms.credit_score < RULES["min_credit_score"]:
        failed_rules.append(f"Credit score {terms.credit_score} below min {RULES['min_credit_score']}")

    if terms.rate_pct > RULES["max_rate_pct"]:
        failed_rules.append(f"Rate {terms.rate_pct}% exceeds max {RULES['max_rate_pct']}%")

    if terms.amount_usd > RULES["max_loan_usd"]:
        failed_rules.append(f"Amount ${terms.amount_usd:,.0f} exceeds max ${RULES['max_loan_usd']:,.0f}")

    if any(b in terms.jurisdiction.upper() for b in RULES["blocked_jurisdictions"]):
        failed_rules.append(f"Jurisdiction '{terms.jurisdiction}' is blocked")

    if terms.amount_usd >= RULES["escalation_usd"]:
        regulatory_notes.append("Amount exceeds escalation threshold — requires senior review")

    if terms.ltv > 0.75:
        regulatory_notes.append("LTV above 75% — enhanced due diligence applies")

    status = "FAIL" if failed_rules else ("CONDITIONAL" if regulatory_notes else "PASS")

    findings = ComplianceFindings(
        status=status,
        failed_rules=failed_rules,
        regulatory_notes=regulatory_notes,
        confidence=1.0,
        reasoning=(
            f"Rule-based check against {len(RULES)} policy thresholds. "
            f"{len(failed_rules)} violation(s) found."
        ),
    )

    print(f"   ✓ Status : {findings.status}  failed={findings.failed_rules or 'none'}")

    return {"specialist_results": [{"specialist": "compliance",
                                    "findings": findings.model_dump(),
                                    "confidence": findings.confidence}]}


def node_financial_analyst(state: DealState) -> dict:
    print("\n📊 [Financial Analyst] Running...")
    terms = state["terms"]

    structured = llm.with_structured_output(FinancialFindings)
    findings   = structured.invoke([
        SystemMessage(content=(
            "You are a senior loan underwriter. Assess the financial health of this application.\n"
            "Benchmarks:\n"
            "  LTV : <0.65 STRONG | 0.65-0.75 ADEQUATE | 0.75-0.80 BORDERLINE | >0.80 WEAK\n"
            "  DTI : <0.36 STRONG | 0.36-0.43 ADEQUATE | 0.43-0.50 WEAK | >0.50 CRITICAL\n"
            "  Credit: >740 STRONG | 700-740 GOOD | 660-700 FAIR | 620-660 MARGINAL\n"
            "Confidence must be a decimal between 0 and 1.\n"
            "Be specific about strengths and concerns."
        )),
        HumanMessage(content=f"Loan terms:\n{json.dumps(terms.model_dump(), indent=2)}"),
    ])

    print(f"   ✓ Financial health : {findings.financial_health}")
    print(f"   ✓ Confidence       : {findings.confidence:.0%}")

    return {"specialist_results": [{"specialist": "financial",
                                    "findings": findings.model_dump(),
                                    "confidence": findings.confidence}]}


def node_risk_quantifier(state: DealState) -> dict:
    print("\n🎲 [Risk Quantifier] Running...")
    terms = state["terms"]

    agent  = create_react_agent(llm, RISK_TOOLS, response_format=RiskFindings)
    result = agent.invoke({"messages": [SystemMessage(content=(
        f"You are a quantitative risk analyst.\n"
        f"Loan terms: {json.dumps(terms.model_dump(), indent=2)}\n\n"
        "Step 1: Call calc_credit_risk with the borrower's metrics.\n"
        "Step 2: Call calc_concentration_risk with the loan amount.\n"
        "Step 3: Composite = 60% credit + 40% concentration.\n"
        "        Risk levels: LOW (<25) MEDIUM (25-50) HIGH (50-75) CRITICAL (>75)."
    ))]})

    findings = result["structured_response"]

    print(f"   ✓ Composite score : {findings.composite_risk_score:.0f}/100 ({findings.risk_level})")
    print(f"   ✓ Confidence      : {findings.confidence:.0%}")

    return {"specialist_results": [{"specialist": "risk",
                                    "findings": findings.model_dump(),
                                    "confidence": findings.confidence}]}


### 7c — Synthesizer, Human Review, Report Generator

In [ ]:
def node_synthesize(state: DealState) -> dict:
    print("\n🔮 [Synthesizer] Aggregating specialist panel...")

    results  = state["specialist_results"]
    terms    = state["terms"]

    by_type  = {r["specialist"]: r["findings"] for r in results}

    avg_conf = sum(r["confidence"] for r in results) / max(len(results), 1)

    structured = llm.with_structured_output(PanelDecision)
    decision   = structured.invoke([
        SystemMessage(content=(
            "You are the chair of a credit committee. Based on the specialist panel, issue a final decision.\n"
            "Decision guide:\n"
            "  APPROVE            — all panels positive, risk <35, compliance PASS\n"
            "  CONDITIONAL_APPROVE — minor issues, conditions can mitigate risk\n"
            "  ESCALATE           — amount >$10M, or avg confidence <0.65, or conflicting signals\n"
            "  REJECT             — compliance FAIL, risk >75, or fundamental underwriting flaws"
        )),
        HumanMessage(content=(
            f"Deal terms:\n{json.dumps(terms.model_dump(), indent=2)}\n\n"
            f"Compliance panel:\n{json.dumps(by_type.get('compliance', {}), indent=2)}\n\n"
            f"Financial panel:\n{json.dumps(by_type.get('financial', {}), indent=2)}\n\n"
            f"Risk panel:\n{json.dumps(by_type.get('risk', {}), indent=2)}\n\n"
            f"Panel average confidence: {avg_conf:.0%}\n"
            "Issue your credit committee decision."
        )),
    ])

    escalate = (
        decision.requires_escalation
        or decision.decision == "ESCALATE"
        or terms.amount_usd >= RULES["escalation_usd"]
        or avg_conf < 0.65
    )

    print(f"   ✓ AI decision : {decision.decision}")
    print(f"   ✓ Risk score  : {decision.overall_risk_score:.0f}/100")
    print(f"   ✓ Confidence  : {decision.panel_confidence:.0%}")
    print(f"   ✓ Escalation  : {escalate}")

    return {"panel_decision": decision, "requires_escalation": escalate}


def node_human_review(state: DealState) -> dict:
    terms    = state["terms"]
    decision = state["panel_decision"]

    packet = {
        "deal_id":    state["deal_id"],
        "borrower":   terms.borrower,
        "amount_usd": terms.amount_usd,
        "ai_decision": decision.decision,
        "risk_score":  decision.overall_risk_score,
        "confidence":  decision.panel_confidence,
        "summary":     decision.executive_summary,
        "key_risks":   decision.key_risks,
        "conditions":  decision.conditions,
        "prompt":      "Enter: APPROVE | REJECT | CONDITIONAL_APPROVE  (optionally add '| notes')",
    }

    print("\n👤 [Human Review] Escalation triggered — pausing for human input...")
    print(json.dumps(packet, indent=2))

    human_input = interrupt(packet)

    if isinstance(human_input, dict):
        h_decision = human_input.get("decision", decision.decision).upper()
        h_notes    = human_input.get("notes", "")
    else:
        parts      = str(human_input).strip().split("|", 1)
        h_decision = parts[0].strip().upper()
        h_notes    = parts[1].strip() if len(parts) > 1 else ""

    print(f"   ✓ Human override: {h_decision}")
    return {"human_decision": h_decision, "human_notes": h_notes}


def node_generate_report(state: DealState) -> dict:
    print("\n📋 [Reporter] Generating audit report...")

    terms    = state["terms"]
    decision = state["panel_decision"]
    results  = state["specialist_results"]

    by_type  = {r["specialist"]: r["findings"] for r in results}

    jev_review = state.get("jev_review") or {}
    final_decision = state.get("human_decision") or decision.decision

    report = {
        "deal_id":   state["deal_id"],
        "timestamp": datetime.datetime.now().isoformat(),

        "deal_summary": terms.model_dump(),

        "specialist_panel": {
            "compliance": by_type.get("compliance"),
            "financial":  by_type.get("financial"),
            "risk":       by_type.get("risk"),
        },

        "ai_recommendation": {
            "decision":          decision.decision,
            "overall_risk":      decision.overall_risk_score,
            "confidence":        decision.panel_confidence,
            "conditions":        decision.conditions,
            "key_risks":         decision.key_risks,
            "executive_summary": decision.executive_summary,
        },

        "final_outcome": {
            "decision":       final_decision,
            "escalated":      state["requires_escalation"],
            "human_override": state.get("human_decision") is not None,
            "human_notes":    state.get("human_notes"),
            "review_source":  "human" if state.get("human_decision") else "panel",
        },
    }

    report["jev_review"] = jev_review or None

    print(f"   ✓ Report ready — {state['deal_id']}")
    return {"final_report": report}


## Step 8 — Build Graph

Every deal goes from `synthesize` to `jev_review`. Jev then routes to human review
or report generation. `MemorySaver` preserves state across the human pause.


In [ ]:
def fan_out_to_specialists(state: DealState) -> list[Send]:
    """After extraction, launch all three specialists at the same time."""
    return [
        Send("compliance_specialist", state),
        Send("financial_analyst",     state),
        Send("risk_quantifier",       state),
    ]


checkpointer = MemorySaver()

builder = StateGraph(DealState)

builder.add_node("extract_terms",          node_extract)
builder.add_node("compliance_specialist",  node_compliance_specialist)
builder.add_node("financial_analyst",      node_financial_analyst)
builder.add_node("risk_quantifier",        node_risk_quantifier)
builder.add_node("synthesize",             node_synthesize)
builder.add_node("jev_review",             node_jev_review)
builder.add_node("human_review",           node_human_review)
builder.add_node("generate_report",        node_generate_report)


builder.add_edge(START, "extract_terms")

builder.add_conditional_edges(
    "extract_terms",
    fan_out_to_specialists,
    ["compliance_specialist", "financial_analyst", "risk_quantifier"],
)

builder.add_edge("compliance_specialist", "synthesize")
builder.add_edge("financial_analyst",     "synthesize")
builder.add_edge("risk_quantifier",       "synthesize")

builder.add_edge("synthesize", "jev_review")

builder.add_conditional_edges("jev_review", route_post_jev, ["human_review", "generate_report"])

builder.add_edge("human_review",    "generate_report")

builder.add_edge("generate_report", END)

graph = builder.compile(checkpointer=checkpointer)

print("graph compiled  — nodes:", list(graph.get_graph().nodes.keys()))


## Step 9 — Visualise Graph

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_ascii())


## Step 10 — Run Pipeline

Each deal has its own checkpoint thread. If the graph pauses, review the packet
and run the optional resume cell with your decision.


In [ ]:
reader    = PdfReader("test_deal.pdf")
deal_text = "\n".join(page.extract_text() or "" for page in reader.pages)

deal_id = f"DEAL-{uuid.uuid4().hex[:6].upper()}"

print(f"PDF loaded  : {len(deal_text)} chars")
print(f"Deal ID     : {deal_id}")
print("=" * 60)

initial: DealState = {
    "deal_id":             deal_id,
    "deal_text":           deal_text,
    "terms":               None,
    "specialist_results":  [],
    "panel_decision":      None,
    "requires_escalation": False,
    "human_decision":      None,
    "human_notes":         None,
    "jev_review":          None,
    "final_report":        None,
}

config = {"configurable": {"thread_id": deal_id}}

print("\n🚀 Starting Parallel Expert Panel...\n")

final = graph.invoke(initial, config=config)
print("\n" + "=" * 60)
if graph.get_state(config).next:
    print("⏸️ Pipeline paused — run the human resume cell below.")
    print(json.dumps(final.get("jev_review"), indent=2))
else:
    print("✅  Pipeline complete")


### (Optional) Resume after human review

Only run this cell if the pipeline paused above.

In [ ]:
HUMAN_DECISION = "CONDITIONAL_APPROVE"
HUMAN_NOTES    = "Escalation reviewed by credit committee. Approved subject to conditions."

final = graph.invoke(
    Command(resume={"decision": HUMAN_DECISION, "notes": HUMAN_NOTES}),
    config=config,
)
print("\n✅  Pipeline resumed and complete")


## Step 11 — Results

In [ ]:
if not final.get("final_report"):
    print("Human review is pending. Resume the pipeline before viewing the report.")
else:
    rpt  = final["final_report"]
    ds   = rpt["deal_summary"]        
    sp   = rpt["specialist_panel"]    
    ai   = rpt["ai_recommendation"]  
    out  = rpt["final_outcome"]      
    
    
    comp = sp.get("compliance") or {}
    fin  = sp.get("financial")  or {}
    risk = sp.get("risk")       or {}
    
    print(f"""
    ╔══════════════════════════════════════════════════════════════╗
    ║               DEAL REVIEW REPORT  —  VERSION JEV                     ║
    ╚══════════════════════════════════════════════════════════════╝
    
    Deal ID    : {rpt['deal_id']}
    Timestamp  : {rpt['timestamp']}
    
    ── DEAL SUMMARY ───────────────────────────────────────────────
    Borrower   : {ds['borrower']}
    Amount     : ${ds['amount_usd']:>15,.0f}
    LTV        : {ds['ltv']:.1%}          (max {RULES['max_ltv']:.0%})
    DTI        : {ds['dti']:.1%}          (max {RULES['max_dti']:.0%})
    Credit     : {ds['credit_score']}           (min {RULES['min_credit_score']})
    Rate       : {ds['rate_pct']}%          (max {RULES['max_rate_pct']}%)
    Term       : {ds['term_months']} months
    Jurisdiction: {ds['jurisdiction']}
    
    ── SPECIALIST PANEL ───────────────────────────────────────────
    ⚖️  Compliance   : {comp.get('status', 'N/A'):20s}  conf {comp.get('confidence', 0):.0%}
    📊 Financial    : {fin.get('financial_health', 'N/A'):20s}  conf {fin.get('confidence', 0):.0%}
    🎲 Risk Score   : {risk.get('composite_risk_score', 0):.0f}/100 ({risk.get('risk_level', 'N/A'):8s})  conf {risk.get('confidence', 0):.0%}
    
    ── AI RECOMMENDATION ──────────────────────────────────────────
    Decision   : {ai['decision']}
    Risk Score : {ai['overall_risk']:.0f}/100
    Confidence : {ai['confidence']:.0%}
    
    ── FINAL OUTCOME ──────────────────────────────────────────────
    ✅ Decision      : {out['decision']}
    👤 Escalated     : {out['escalated']}
    🔁 Human Override: {out['human_override']}
    Reviewer        : {out['review_source']}
    """)
    
    if out.get("human_notes"):
        print(f"Human notes  : {out['human_notes']}\n")
    
    if ai.get("conditions"):
        print("Conditions:")
        for c in ai["conditions"]:
            print(f"  • {c}")
    
    if ai.get("key_risks"):
        print("\nKey Risks:")
        for r in ai["key_risks"]:
            print(f"  ⚠  {r}")
    
    print(f"\nExecutive Summary:\n{ai['executive_summary']}")
    if rpt.get("jev_review"):
        print("\nJev review:", json.dumps(rpt["jev_review"], indent=2))


## Step 12 — Inspect Checkpointed State

Every state transition is stored in `MemorySaver`. You can replay or audit any checkpoint.

In [ ]:
history = list(graph.get_state_history(config))

print(f"Checkpoints saved: {len(history)}\n")
for i, snapshot in enumerate(reversed(history)):
    node = snapshot.metadata.get("source", "?") if snapshot.metadata else "?"
    keys = [k for k, v in snapshot.values.items() if v is not None]
    print(f"  [{i:02d}] next={snapshot.next}  populated_keys={keys}")
